# Practice 1.2 - Vietnamese Word Segmentation Solution

This notebook mirrors the `solution/word_segmenter.py` implementation and keeps the same forward maximum matching behavior.


## Table of Contents

1. Setup and data paths.
2. Unicode normalization and tokenization helpers.
3. Dictionary loading, segmentation, and evaluation.
4. Quick experiments on one sentence and the 100-sentence benchmark.


In [1]:
from pathlib import Path
import unicodedata
from typing import Iterable

def resolve_base_dir() -> Path:
    markers = (
        Path('vn-dict.txt'),
        Path('data/eval_input.txt'),
        Path('data/eval_gold.txt'),
    )
    cwd = Path.cwd().resolve()

    for search_root in (cwd, *cwd.parents):
        for candidate in (search_root, search_root / 'Practice/1-2 Word Segmentation'):
            if all((candidate / marker).exists() for marker in markers):
                return candidate

    raise FileNotFoundError(
        'Could not locate Practice/1-2 Word Segmentation from the current working directory.'
    )

base_dir = resolve_base_dir()

dict_path = base_dir / 'vn-dict.txt'
eval_input_path = base_dir / 'data' / 'eval_input.txt'
eval_gold_path = base_dir / 'data' / 'eval_gold.txt'
eval_pred_path = base_dir / 'data' / 'eval_pred_from_notebook.txt'

print(f'Base directory: {base_dir.resolve()}')


Base directory: /home/tung/TA_NLP-1/Practice/1-2 Word Segmentation


In [2]:
EDGE_PUNCTUATION = set("\"'“”‘’.,!?;:()[]{}|-")

def normalize_unicode(text: str) -> str:
    return unicodedata.normalize('NFC', text)

def normalize_whitespace(text: str) -> str:
    return ' '.join(normalize_unicode(text).split())

def normalize_lookup(text: str) -> str:
    return normalize_whitespace(text).lower()

def split_edge_punctuation(token: str) -> list[str]:
    if not token:
        return []

    leading: list[str] = []
    trailing: list[str] = []
    core = token

    while core and core[0] in EDGE_PUNCTUATION:
        leading.append(core[0])
        core = core[1:]

    while core and core[-1] in EDGE_PUNCTUATION:
        trailing.append(core[-1])
        core = core[:-1]

    parts = leading
    if core:
        parts.append(core)
    parts.extend(reversed(trailing))
    return parts

def tokenize_text(text: str) -> list[str]:
    tokens: list[str] = []
    for raw_token in normalize_whitespace(text).split():
        parts = split_edge_punctuation(raw_token)
        tokens.extend(parts if parts else [raw_token])
    return tokens


In [3]:
def load_dictionary(path: str | Path) -> tuple[set[str], int]:
    lexicon: set[str] = set()
    max_word_len = 1

    for line in Path(path).read_text(encoding='utf-8').splitlines():
        entry = normalize_whitespace(line)
        if not entry:
            continue

        lookup_key = entry.lower()
        if lookup_key in lexicon:
            continue

        lexicon.add(lookup_key)
        max_word_len = max(max_word_len, len(lookup_key.split()))

    return lexicon, max_word_len

def segment_tokens(tokens: list[str], lexicon: set[str], max_word_len: int) -> list[str]:
    segmented: list[str] = []
    index = 0

    while index < len(tokens):
        best_match_len = 0
        window = min(max_word_len, len(tokens) - index)

        for size in range(window, 0, -1):
            candidate = ' '.join(tokens[index : index + size])
            if normalize_lookup(candidate) in lexicon:
                best_match_len = size
                break

        if best_match_len == 0:
            segmented.append(tokens[index])
            index += 1
            continue

        segmented.append('_'.join(tokens[index : index + best_match_len]))
        index += best_match_len

    return segmented

def segment_text(text: str, lexicon: set[str], max_word_len: int) -> list[str]:
    return segment_tokens(tokenize_text(text), lexicon, max_word_len)


In [4]:
def sentence_to_boundaries(sentence: str) -> tuple[list[str], set[tuple[int, int]]]:
    syllables: list[str] = []
    boundaries: set[tuple[int, int]] = set()
    cursor = 0

    for token in normalize_whitespace(sentence).split():
        parts = token.split('_')
        if any(part == '' for part in parts):
            raise ValueError(f'Invalid token in segmented sentence: {token!r}')

        normalized_parts = [normalize_unicode(part) for part in parts]
        syllables.extend(normalized_parts)
        boundaries.add((cursor, cursor + len(normalized_parts)))
        cursor += len(normalized_parts)

    comparable_syllables = [normalize_lookup(part) for part in syllables]
    return comparable_syllables, boundaries

def evaluate(pred_sentences: Iterable[str], gold_sentences: Iterable[str]) -> tuple[float, float, float]:
    pred_list = list(pred_sentences)
    gold_list = list(gold_sentences)

    if len(pred_list) != len(gold_list):
        raise ValueError('Prediction and gold files must have the same number of lines.')

    matched = 0
    predicted_total = 0
    gold_total = 0

    for line_number, (pred_sentence, gold_sentence) in enumerate(zip(pred_list, gold_list), start=1):
        pred_syllables, pred_boundaries = sentence_to_boundaries(pred_sentence)
        gold_syllables, gold_boundaries = sentence_to_boundaries(gold_sentence)

        if pred_syllables != gold_syllables:
            raise ValueError(
                'Prediction and gold sentence differ after removing segmentation markers '
                f'at line {line_number}.'
            )

        matched += len(pred_boundaries & gold_boundaries)
        predicted_total += len(pred_boundaries)
        gold_total += len(gold_boundaries)

    precision = matched / predicted_total if predicted_total else 0.0
    recall = matched / gold_total if gold_total else 0.0
    f1 = 0.0 if precision + recall == 0.0 else 2 * precision * recall / (precision + recall)
    return precision, recall, f1

def read_lines(path: str | Path) -> list[str]:
    return Path(path).read_text(encoding='utf-8').splitlines()

def write_lines(path: str | Path, lines: Iterable[str]) -> None:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')


In [5]:
lexicon, max_word_len = load_dictionary(dict_path)
print(f'Dictionary entries: {len(lexicon):,}')
print(f'Max syllables in a dictionary entry: {max_word_len}')


Dictionary entries: 29,198
Max syllables in a dictionary entry: 14


In [8]:
example = 'Ngày mai khác hôm nay  .'
print('Input :', example)
print('Output:', ' '.join(segment_text(example, lexicon, max_word_len)))


Input : Ngày mai khác hôm nay  .
Output: Ngày_mai khác hôm_nay .


In [7]:
predictions = [' '.join(segment_text(line, lexicon, max_word_len)) for line in read_lines(eval_input_path)]
write_lines(eval_pred_path, predictions)
precision, recall, f1 = evaluate(predictions, read_lines(eval_gold_path))
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1: {f1:.4f}')
print(f'Predictions written to: {eval_pred_path}')


Precision: 0.7741
Recall: 0.8698
F1: 0.8192
Predictions written to: /home/tung/TA_NLP-1/Practice/1-2 Word Segmentation/data/eval_pred_from_notebook.txt
